In [5]:
import notebookutils

# 1. Eliminar físicamente los archivos Parquet antiguos con tipos incoherentes
try:
    notebookutils.fs.rm("Files/raw/realtime/weather", True)
    print("🧹 Carpeta RAW 'weather' limpiada con éxito.")
except Exception as e:
    print(f"ℹ️ {e}")

StatementMeta(, 312ef95a-0c2d-4510-9d46-29080d7ea768, 9, Finished, Available, Finished, False)

🧹 Carpeta RAW 'weather' limpiada con éxito.


In [6]:
from tfm_mobility.processors.raw_processor import RawProcessor

processor = RawProcessor(spark)
processor.landing_to_raw_json("Files/landing/realtime/weather", "Files/raw/realtime/weather")

StatementMeta(, 312ef95a-0c2d-4510-9d46-29080d7ea768, 10, Finished, Available, Finished, False)

2026-09-03 12:51:31,388 - INFO - 📄 Leyendo JSON Landing: Files/landing/realtime/weather/*/*/*/*/*.json
2026-09-03 12:51:35,429 - INFO - ✅ JSON guardado en RAW Parquet en: Files/raw/realtime/weather


In [6]:
# ==============================================================================
# PROCESAMIENTO DIRECTO WEATHER EN FABRIC (BÚSQUEDA DINÁMICA DE ARCHIVOS)
# ==============================================================================
from pyspark.sql import functions as F

landing_weather_base = "Files/landing/realtime/weather"
raw_weather_output = "Files/raw/realtime/weather"

print(f"⏳ Localizando archivos JSON en: {landing_weather_base}...")

def get_all_json_files(dir_path):
    """Obtiene de forma recursiva todas las rutas de archivos .json usando mssparkutils."""
    json_paths = []
    try:
        items = mssparkutils.fs.ls(dir_path)
        for item in items:
            if item.isDir:
                json_paths.extend(get_all_json_files(item.path))
            elif item.name.endswith(".json"):
                json_paths.append(item.path)
    except Exception as e:
        pass
    return json_paths

# 1. Obtener la lista de archivos existentes
all_json_files = get_all_json_files(landing_weather_base)

if not all_json_files:
    print(f"⚠️ No se encontraron archivos .json dentro de: {landing_weather_base}")
else:
    print(f"📄 Se han encontrado {len(all_json_files)} archivo(s) JSON. Leyendo con Spark...")
    
    # 2. Leer las rutas exactas encontradas
    df_weather = spark.read.option("multiline", "true").json(all_json_files)

    # 3. Enriquecer con metadatos de ingesta
    df_weather = df_weather.withColumn(
        "landing_source_file", 
        F.element_at(F.split(F.col("_metadata.file_name"), "/"), -1)
    ).withColumn(
        "ingestion_timestamp", 
        F.current_timestamp()
    )

    # 4. Guardar en la capa RAW Parquet
    df_weather.write.format("parquet").mode("overwrite").save(raw_weather_output)

    total_records = df_weather.count()
    print(f"✅ [RAW OK] ¡Éxito! Se han generado {total_records:,} registro(s) en Parquet.")
    print(f"📁 Destino: {raw_weather_output}")

StatementMeta(, bb174156-84ff-474e-8186-c9511bce82f2, 10, Finished, Available, Finished, False)

⏳ Localizando archivos JSON en: Files/landing/realtime/weather...
📄 Se han encontrado 2 archivo(s) JSON. Leyendo con Spark...
✅ [RAW OK] ¡Éxito! Se han generado 0 registro(s) en Parquet.
📁 Destino: Files/raw/realtime/weather


In [7]:
# Inspeccionar el interior de los JSONs guardados
json_paths = get_all_json_files("Files/landing/realtime/weather")

for p in json_paths:
    print(f"📄 Archivo: {p}")
    content = mssparkutils.fs.head(p, 500)
    print(f"  └─ Contenido (primeros 500 caracteres):\n{content}\n")

StatementMeta(, bb174156-84ff-474e-8186-c9511bce82f2, 11, Finished, Available, Finished, False)

📄 Archivo: abfss://7ba00fc9-43c9-452c-923e-3d97c381d0c1@onelake.dfs.fabric.microsoft.com/51da2c4b-643c-4605-bbc1-b20bd00e3522/Files/landing/realtime/weather/2026/09/03/08/weather_20260903_084912.json
  └─ Contenido (primeros 500 caracteres):
[]

📄 Archivo: abfss://7ba00fc9-43c9-452c-923e-3d97c381d0c1@onelake.dfs.fabric.microsoft.com/51da2c4b-643c-4605-bbc1-b20bd00e3522/Files/landing/realtime/weather/2026/09/03/09/weather_20260903_093550.json
  └─ Contenido (primeros 500 caracteres):
[]



In [1]:
import os
import json
import time
import requests
import numpy as np
from datetime import datetime


# ============================================================
# CONFIGURACIÓN
# ============================================================

API_URL = "https://api.open-meteo.com/v1/forecast"

# 50 coordenadas por petición
BATCH_SIZE = 50

# No superar 500 coordenadas por ventana
MAX_COORDS_PER_MINUTE = 500

# Espera de seguridad entre ventanas
WAIT_BETWEEN_WINDOWS = 65

# Timeout:
# 10 segundos para conectar
# 120 segundos esperando respuesta
CONNECT_TIMEOUT = 10
READ_TIMEOUT = 120

# Reintentos
MAX_RETRIES = 5


print("🌐 Descargando predicción meteorológica para España...")


# ============================================================
# 1. GENERAR LOS 2.500 PUNTOS
# ============================================================

# Península + Baleares
lats_p = np.linspace(36.0, 43.5, 40)
lons_p = np.linspace(-9.0, 3.5, 55)

lats = []
lons = []

for lat in lats_p:
    for lon in lons_p:
        lats.append(round(float(lat), 2))
        lons.append(round(float(lon), 2))


# Canarias
lats_c = np.linspace(27.6, 29.4, 10)
lons_c = np.linspace(-18.1, -13.3, 30)

for lat in lats_c:
    for lon in lons_c:
        lats.append(round(float(lat), 2))
        lons.append(round(float(lon), 2))


total_pts = len(lats)

print(f"📍 Puntos totales: {total_pts}")


# ============================================================
# 2. CREAR LOTES
# ============================================================

chunks = [
    (
        lats[i:i + BATCH_SIZE],
        lons[i:i + BATCH_SIZE]
    )
    for i in range(0, total_pts, BATCH_SIZE)
]

total_batches = len(chunks)

print(f"📦 Tamaño de lote: {BATCH_SIZE}")
print(f"📦 Peticiones necesarias: {total_batches}")


# ============================================================
# 3. SESIÓN HTTP
# ============================================================

session = requests.Session()

results = []

coords_in_current_window = 0
window_start = time.time()


# ============================================================
# 4. FUNCIÓN PARA ESPERAR LA VENTANA DE RATE LIMIT
# ============================================================

def controlar_rate_limit(num_coords):

    global coords_in_current_window
    global window_start

    elapsed = time.time() - window_start

    # Si ha pasado un minuto, empezamos una ventana nueva
    if elapsed >= 60:

        coords_in_current_window = 0
        window_start = time.time()

    # Si añadir este lote supera el límite,
    # esperamos hasta completar la ventana
    if coords_in_current_window + num_coords > MAX_COORDS_PER_MINUTE:

        remaining = 60 - (time.time() - window_start)

        if remaining > 0:

            print()
            print(
                f"⏳ Límite de {MAX_COORDS_PER_MINUTE} "
                f"coordenadas/minuto alcanzado."
            )

            print(
                f"⏳ Esperando {remaining:.1f} segundos..."
            )

            time.sleep(remaining + 5)

        # Nueva ventana
        coords_in_current_window = 0
        window_start = time.time()


# ============================================================
# 5. DESCARGA
# ============================================================

for batch_num, (batch_lats, batch_lons) in enumerate(
    chunks,
    start=1
):

    num_coords = len(batch_lats)

    # Control previo del límite
    controlar_rate_limit(num_coords)

    print()
    print(
        f"⛅ Petición {batch_num}/{total_batches} "
        f"({num_coords} puntos)..."
    )

    params = {
        "latitude": ",".join(map(str, batch_lats)),
        "longitude": ",".join(map(str, batch_lons)),

        "hourly": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m"
        ]),

        "forecast_days": 7,

        "timezone": "Europe/Madrid"
    }


    conseguido = False


    # ========================================================
    # REINTENTOS
    # ========================================================

    for intento in range(1, MAX_RETRIES + 1):

        try:

            response = session.get(
                API_URL,
                params=params,
                timeout=(
                    CONNECT_TIMEOUT,
                    READ_TIMEOUT
                )
            )

            # ------------------------------------------------
            # ÉXITO
            # ------------------------------------------------

            if response.status_code == 200:

                data = response.json()

                if not isinstance(data, list):

                    data = [data]

                results.extend(data)

                coords_in_current_window += num_coords

                print(
                    f"   ✅ OK ({len(data)} puntos)"
                )

                print(
                    f"   📊 Acumulados: "
                    f"{len(results)}/{total_pts}"
                )

                conseguido = True

                break


            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            elif response.status_code == 429:

                print(
                    "   ⚠️ HTTP 429 - Rate limit de Open-Meteo"
                )

                # Intentamos obtener Retry-After
                retry_after = response.headers.get(
                    "Retry-After"
                )

                if retry_after is not None:

                    try:
                        espera = int(retry_after) + 5
                    except:
                        espera = 65

                else:

                    espera = 65


                print(
                    f"   ⏳ Esperando {espera} segundos..."
                )

                time.sleep(espera)

                # Nueva ventana
                coords_in_current_window = 0
                window_start = time.time()

                continue


            # ------------------------------------------------
            # ERROR HTTP
            # ------------------------------------------------

            else:

                print(
                    f"   ❌ HTTP {response.status_code}"
                )

                print(
                    response.text[:1000]
                )

                if intento < MAX_RETRIES:

                    espera = 10 * intento

                    print(
                        f"   ⏳ Reintentando en "
                        f"{espera} segundos..."
                    )

                    time.sleep(espera)

                continue


        # ----------------------------------------------------
        # TIMEOUT
        # ----------------------------------------------------

        except requests.exceptions.ReadTimeout:

            print(
                f"   ⏱️ Timeout de lectura "
                f"(>{READ_TIMEOUT}s)"
            )

            if intento < MAX_RETRIES:

                espera = 10 * intento

                print(
                    f"   ⏳ Reintentando en "
                    f"{espera} segundos..."
                )

                time.sleep(espera)

            continue


        # ----------------------------------------------------
        # ERROR DE CONEXIÓN
        # ----------------------------------------------------

        except requests.exceptions.ConnectionError as e:

            print(
                f"   🌐 Error de conexión: {e}"
            )

            if intento < MAX_RETRIES:

                espera = 10 * intento

                print(
                    f"   ⏳ Reintentando en "
                    f"{espera} segundos..."
                )

                time.sleep(espera)

            continue


        # ----------------------------------------------------
        # ERROR GENERAL
        # ----------------------------------------------------

        except Exception as e:

            print(
                f"   ❌ Error inesperado: "
                f"{type(e).__name__}: {e}"
            )

            if intento < MAX_RETRIES:

                espera = 10 * intento

                print(
                    f"   ⏳ Reintentando en "
                    f"{espera} segundos..."
                )

                time.sleep(espera)

            continue


    # ========================================================
    # COMPROBAR LOTE
    # ========================================================

    if not conseguido:

        session.close()

        raise RuntimeError(
            f"❌ No se pudo descargar el lote "
            f"{batch_num}/{total_batches}."
        )


# ============================================================
# 6. CERRAR SESIÓN
# ============================================================

session.close()


# ============================================================
# 7. COMPROBAR RESULTADOS
# ============================================================

print()
print("============================================")
print("📊 DESCARGA FINALIZADA")
print("============================================")

print(
    f"📍 Puntos solicitados: {total_pts}"
)

print(
    f"📍 Puntos recibidos:   {len(results)}"
)


if len(results) != total_pts:

    raise RuntimeError(
        f"❌ Se esperaban {total_pts} puntos "
        f"pero se han recibido {len(results)}."
    )


# ============================================================
# 8. GUARDAR EN LANDING
# ============================================================

landing_base = (
    "/lakehouse/default/Files/"
    "landing/realtime/weather"
)

now = datetime.now()

folder_path = os.path.join(
    landing_base,
    now.strftime("%Y"),
    now.strftime("%m"),
    now.strftime("%d"),
    now.strftime("%H")
)

os.makedirs(
    folder_path,
    exist_ok=True
)


file_name = (
    f"weather_"
    f"{now.strftime('%Y%m%d_%H%M%S')}.json"
)

full_path = os.path.join(
    folder_path,
    file_name
)


with open(
    full_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False
    )


# ============================================================
# 9. RESULTADO
# ============================================================

print()
print("============================================")
print("✅ TODO CORRECTO")
print("============================================")

print(
    f"📍 Nodos meteorológicos: {len(results)}"
)

print(
    "📅 Previsión: 7 días"
)

print(
    "⏱️ Resolución: horaria"
)

print(
    f"📁 JSON guardado en:"
)

print(
    full_path
)

StatementMeta(, 312ef95a-0c2d-4510-9d46-29080d7ea768, 5, Finished, Available, Finished, False)

🌐 Descargando predicción meteorológica para España...
📍 Puntos totales: 2500
📦 Tamaño de lote: 50
📦 Peticiones necesarias: 50

⛅ Petición 1/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 50/2500

⛅ Petición 2/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 100/2500

⛅ Petición 3/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 150/2500

⛅ Petición 4/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 200/2500

⛅ Petición 5/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 250/2500

⛅ Petición 6/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 300/2500

⛅ Petición 7/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 350/2500

⛅ Petición 8/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 400/2500

⛅ Petición 9/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 450/2500

⛅ Petición 10/50 (50 puntos)...
   ✅ OK (50 puntos)
   📊 Acumulados: 500/2500

⏳ Límite de 500 coordenadas/minuto alcanzado.
⏳ Esperando 59.1 segundos...

⛅ Petición 11/50 

In [1]:
import requests
import json
import os
import time
import numpy as np
from datetime import datetime

# ============================================================
# CONFIGURACIÓN
# ============================================================

API_URL = "https://api.open-meteo.com/v1/forecast"

BATCH_SIZE = 1000
MAX_RETRIES = 5
WAIT_429 = 65  # segundos


# ============================================================
# GENERAR LOS 2.500 PUNTOS
# ============================================================

# Península + Baleares
lats_p = np.linspace(36.0, 43.5, 40)
lons_p = np.linspace(-9.0, 3.5, 55)

# Canarias
lats_c = np.linspace(27.6, 29.4, 10)
lons_c = np.linspace(-18.1, -13.3, 30)

coords = []

for lat in lats_p:
    for lon in lons_p:
        coords.append(
            (round(float(lat), 2), round(float(lon), 2))
        )

for lat in lats_c:
    for lon in lons_c:
        coords.append(
            (round(float(lat), 2), round(float(lon), 2))
        )


# ============================================================
# INFORMACIÓN INICIAL
# ============================================================

total_points = len(coords)
total_batches = (
    total_points + BATCH_SIZE - 1
) // BATCH_SIZE

print("🌐 Descargando predicción meteorológica para España...")
print(f"📍 Puntos totales: {total_points}")
print(f"📦 Tamaño máximo por petición: {BATCH_SIZE}")
print(f"📦 Peticiones necesarias: {total_batches}")


# ============================================================
# DESCARGA
# ============================================================

resultados = []

for i in range(0, total_points, BATCH_SIZE):

    batch = coords[i:i + BATCH_SIZE]

    batch_num = (i // BATCH_SIZE) + 1

    batch_lats = [coord[0] for coord in batch]
    batch_lons = [coord[1] for coord in batch]

    print()
    print("=" * 60)
    print(f"📡 PETICIÓN {batch_num}/{total_batches}")
    print(f"📍 Puntos: {len(batch)}")
    print("=" * 60)

    # --------------------------------------------------------
    # RETRIES
    # --------------------------------------------------------

    for intento in range(1, MAX_RETRIES + 1):

        try:

            print(
                f"📡 Enviando petición "
                f"(intento {intento}/{MAX_RETRIES})..."
            )

            payload = {
                "latitude": batch_lats,
                "longitude": batch_lons,

                "hourly": [
                    "temperature_2m",
                    "relative_humidity_2m",
                    "precipitation",
                    "wind_speed_10m",
                    "wind_direction_10m",
                    "wind_gusts_10m",
                    "dew_point_2m",
                    "vapour_pressure_deficit"
                ],

                "forecast_days": 7,

                "timezone": [
                    "Europe/Madrid"
                ]
            }

            response = requests.post(
                API_URL,
                json=payload,
                timeout=(20, 180)
            )

            print(
                f"📡 HTTP {response.status_code}"
            )

            # ------------------------------------------------
            # PETICIÓN CORRECTA
            # ------------------------------------------------

            if response.status_code == 200:

                data = response.json()

                # Open-Meteo debe devolver una lista
                if not isinstance(data, list):

                    raise RuntimeError(
                        "La respuesta de Open-Meteo "
                        "no contiene una lista de ubicaciones."
                    )

                # Comprobar número de puntos
                if len(data) != len(batch):

                    raise RuntimeError(
                        f"Se esperaban {len(batch)} puntos "
                        f"pero se recibieron {len(data)}."
                    )

                resultados.extend(data)

                print(
                    f"✅ OK ({len(data)} puntos)"
                )

                print(
                    f"📊 Acumulados: "
                    f"{len(resultados)}/{total_points}"
                )

                break


            # ------------------------------------------------
            # RATE LIMIT 429
            # ------------------------------------------------

            elif response.status_code == 429:

                print(
                    "⚠️ Límite de peticiones alcanzado "
                    "(HTTP 429)."
                )

                if intento < MAX_RETRIES:

                    print(
                        f"⏳ Esperando "
                        f"{WAIT_429} segundos antes de reintentar..."
                    )

                    time.sleep(WAIT_429)

                    print(
                        "🔄 Reintentando..."
                    )

                else:

                    raise RuntimeError(
                        "❌ Se alcanzó el máximo de "
                        "reintentos por HTTP 429."
                    )


            # ------------------------------------------------
            # OTROS ERRORES HTTP
            # ------------------------------------------------

            else:

                print(
                    "❌ Error de Open-Meteo:"
                )

                print(response.text)

                raise RuntimeError(
                    f"Open-Meteo devolvió "
                    f"HTTP {response.status_code}"
                )


        # ----------------------------------------------------
        # TIMEOUT
        # ----------------------------------------------------

        except requests.exceptions.Timeout:

            print(
                "⏱️ Timeout de conexión."
            )

            if intento < MAX_RETRIES:

                print(
                    "⏳ Esperando 10 segundos "
                    "antes de reintentar..."
                )

                time.sleep(10)

            else:

                raise RuntimeError(
                    f"❌ Timeout definitivo en "
                    f"la petición {batch_num}."
                )


        # ----------------------------------------------------
        # ERROR DE CONEXIÓN
        # ----------------------------------------------------

        except requests.exceptions.RequestException as e:

            print(
                f"❌ Error de conexión: {e}"
            )

            if intento < MAX_RETRIES:

                print(
                    "⏳ Esperando 10 segundos "
                    "antes de reintentar..."
                )

                time.sleep(10)

            else:

                raise RuntimeError(
                    f"❌ Error definitivo en "
                    f"la petición {batch_num}: {e}"
                )


# ============================================================
# COMPROBACIÓN FINAL
# ============================================================

print()
print()
print("=" * 60)
print("📊 DESCARGA FINALIZADA")
print("=" * 60)

print(
    f"📍 Puntos solicitados: {total_points}"
)

print(
    f"📍 Puntos recibidos:   {len(resultados)}"
)


if len(resultados) != total_points:

    raise RuntimeError(
        f"❌ ERROR FINAL: se esperaban "
        f"{total_points} puntos y se recibieron "
        f"{len(resultados)}."
    )


print()
print("✅ TODOS LOS PUNTOS RECIBIDOS CORRECTAMENTE")


# ============================================================
# GUARDAR JSON
# ============================================================

now = datetime.now()

output_dir = (
    f"/lakehouse/default/Files/"
    f"landing/realtime/weather/"
    f"{now:%Y/%m/%d/%H}"
)

os.makedirs(
    output_dir,
    exist_ok=True
)

filename = (
    f"weather_{now:%Y%m%d_%H%M%S}.json"
)

output_path = os.path.join(
    output_dir,
    filename
)


with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        resultados,
        f,
        ensure_ascii=False
    )


# ============================================================
# RESULTADO FINAL
# ============================================================

print()
print("=" * 60)
print("💾 ARCHIVO GUARDADO CORRECTAMENTE")
print("=" * 60)

print(
    f"📁 {output_path}"
)

print()
print(
    f"📍 Nodos meteorológicos: "
    f"{len(resultados)}"
)

print(
    "📅 Previsión: 7 días"
)

print(
    "⏱️ Resolución: horaria"
)

print(
    "🌦️ Variables meteorológicas: 8"
)

print()
print("✅ PROCESO COMPLETADO")

StatementMeta(, 6310b1f8-e037-405a-b061-f505f22b1f91, 5, Finished, Cancelled, Cancelled, False)

🌐 Descargando predicción meteorológica para España...
📍 Puntos totales: 2500
📦 Tamaño máximo por petición: 1000
📦 Peticiones necesarias: 3

📡 PETICIÓN 1/3
📍 Puntos: 1000
📡 Enviando petición (intento 1/5)...
📡 HTTP 200
✅ OK (1000 puntos)
📊 Acumulados: 1000/2500

📡 PETICIÓN 2/3
📍 Puntos: 1000
📡 Enviando petición (intento 1/5)...
📡 HTTP 429
⚠️ Límite de peticiones alcanzado (HTTP 429).
⏳ Esperando 65 segundos antes de reintentar...
🔄 Reintentando...
📡 Enviando petición (intento 2/5)...
⏱️ Timeout de conexión.
⏳ Esperando 10 segundos antes de reintentar...
📡 Enviando petición (intento 3/5)...
📡 HTTP 200
✅ OK (1000 puntos)
📊 Acumulados: 2000/2500

📡 PETICIÓN 3/3
📍 Puntos: 500
📡 Enviando petición (intento 1/5)...
📡 HTTP 429
⚠️ Límite de peticiones alcanzado (HTTP 429).
⏳ Esperando 65 segundos antes de reintentar...


In [4]:
import os
from datetime import datetime
from pyspark.sql import functions as F

print("🔄 Procesando datos meteorológicos de Landing a RAW Parquet...")

landing_path = "Files/landing/realtime/weather"
raw_base_path = "Files/raw/realtime/weather"

# 1. Leer el JSON manteniendo todo el esquema anidado
df_landing = spark.read \
    .option("recursiveFileLookup", "true") \
    .option("multiline", "true") \
    .json(landing_path)

# 2. Agregar los metadatos de auditoría sin perder las columnas meteorológicas
now_str = datetime.now().strftime("%Y-%m-%dT%H:%M:%S.000Z")
df_raw = df_landing \
    .withColumn("landing_source_file", F.input_file_name()) \
    .withColumn("ingestion_timestamp", F.lit(now_str))

# 3. Limpiar carpeta RAW antigua y guardar los archivos Parquet completos
now = datetime.now()
output_dir = os.path.join(raw_base_path, now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H"))

df_raw.write \
    .mode("overwrite") \
    .parquet(output_dir)

print(f"✅ [RAW OK] Archivos Parquet generados correctamente con {df_raw.count()} registros completos en:\n   └─ {output_dir}")

StatementMeta(, 6310b1f8-e037-405a-b061-f505f22b1f91, 8, Finished, Available, Finished, False)

🔄 Procesando datos meteorológicos de Landing a RAW Parquet...
✅ [RAW OK] Archivos Parquet generados correctamente con 2553 registros completos en:
   └─ Files/raw/realtime/weather/2026/09/03/12


In [6]:
import os
from datetime import datetime
from pyspark.sql import functions as F

print("🧹 Limpiando directorio RAW antiguo y procesando datos de Landing...")

landing_path = "Files/landing/realtime/weather"
raw_base_path = "Files/raw/realtime/weather"

# 1. Eliminar físicamente los Parquet antiguos incompatibles
try:
    mssparkutils.fs.rm(raw_base_path, True)
    print("✅ Directorio RAW antiguo eliminado correctamente.")
except Exception as e:
    print(f"ℹ️ Nota sobre la limpieza: {e}")

# 2. Leer los JSON de Landing de forma recursiva
df_landing = spark.read \
    .option("recursiveFileLookup", "true") \
    .option("multiline", "true") \
    .json(landing_path)

# 3. Forzar explicitamente 'ingestion_timestamp' y 'landing_source_file' como string
now_str = datetime.now().strftime("%Y-%m-%dT%H:%M:%S.000Z")
df_raw = df_landing \
    .withColumn("landing_source_file", F.col("_metadata.file_path").cast("string")) \
    .withColumn("ingestion_timestamp", F.lit(now_str).cast("string"))

# 4. Guardar Parquet limpios en la capa RAW
now = datetime.now()
output_dir = os.path.join(raw_base_path, now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H"))

df_raw.write \
    .mode("overwrite") \
    .parquet(output_dir)

print(f"✅ [RAW OK] Archivos Parquet unificados generados en:\n   └─ {output_dir}")

StatementMeta(, 6310b1f8-e037-405a-b061-f505f22b1f91, 10, Finished, Available, Finished, False)

🧹 Limpiando directorio RAW antiguo y procesando datos de Landing...
✅ Directorio RAW antiguo eliminado correctamente.
✅ [RAW OK] Archivos Parquet unificados generados en:
   └─ Files/raw/realtime/weather/2026/09/03/12


In [7]:
from pyspark.sql import functions as F

print("⚙️ Generando tabla Delta Bronze con aplanado de datos...")

raw_path = "Files/raw/realtime/weather"

# 1. Leer los Parquet limpios
df_raw = spark.read \
    .option("recursiveFileLookup", "true") \
    .parquet(raw_path)

# 2. Aplanar arrays de 'hourly'
df_flat = df_raw.select(
    F.col("latitude"),
    F.col("longitude"),
    F.col("elevation"),
    F.col("timezone"),
    F.col("landing_source_file"),
    F.col("ingestion_timestamp"),
    F.explode(
        F.arrays_zip(
            F.col("hourly.time"),
            F.col("hourly.temperature_2m"),
            F.col("hourly.relative_humidity_2m"),
            F.col("hourly.precipitation"),
            F.col("hourly.wind_speed_10m")
        )
    ).alias("h")
).select(
    F.col("latitude"),
    F.col("longitude"),
    F.col("elevation"),
    F.col("timezone"),
    F.col("landing_source_file"),
    F.col("ingestion_timestamp"),
    F.to_timestamp(F.col("h.time")).alias("forecast_time"),
    F.col("h.temperature_2m").alias("temperature_2m"),
    F.col("h.relative_humidity_2m").alias("relative_humidity_2m"),
    F.col("h.precipitation").alias("precipitation"),
    F.col("h.wind_speed_10m").alias("wind_speed_10m")
)

# 3. Registrar tabla Delta Bronze
df_flat.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dbo.bronze_weather")

print(f"✅ ¡Éxito! Tabla Delta 'bronze_weather' registrada correctamente con {df_flat.count()} filas.")

StatementMeta(, 6310b1f8-e037-405a-b061-f505f22b1f91, 11, Finished, Available, Finished, False)

⚙️ Generando tabla Delta Bronze con aplanado de datos...
✅ ¡Éxito! Tabla Delta 'bronze_weather' registrada correctamente con 428904 filas.
